# OneBill Full Migration — Accounts + Subscriptions

Consolidates the two source notebooks into a single, idempotent pipeline:

1. **Customer Migration** — creates the OneBill account (+ contacts)
2. **Subscription Migration** — creates the OneBill order/subscription

## Per-account flow

| Step | What happens |
|---|---|
| 1 | Create the account in OneBill. |
| 1a | If it succeeds → continue. |
| 1b | If OneBill reports **"Account number ... already exists"** → this is **not** treated as a failure, we note the account already existed and continue to Step 2. |
| 1c | Any other error → a real failure; subscription creation for that account is skipped. |
| 2 | For every subscription belonging to this account (matched on `AccountCode`), look up the account's default shipping address in OneBill and create the order. |

This means the notebook is **safe to re-run**: accounts that already exist in OneBill are not re-created, but the notebook will still attempt to create any subscriptions that are missing.

## Sources (unchanged from the originals)

| Data | Source | Purpose in payload |
|---|---|---|
| **Accounts** | MySQL `bi_datastore.billing_account` (`_DataSource = 'vBill'`) | `accountNumber`, `accountName`, `address`, `accountAttribute` |
| **Contacts** | Microsoft Dataverse (Dynamics) `contact` entity via FetchXML | `contact[]` under each account |
| **Subscriptions** | MySQL `bi_curated_views.reporting_subscription` | `orderElement[]` for each account's order |

Both account and subscription creation share **one** OneBill OAuth token manager and **one** `requests.Session` / connection pool.

## Execution order

| Section | What happens |
|---|---|
| 1 | Imports and config |
| 2 | Dataverse OAuth + FetchXML fetch (contacts) |
| 3 | Contact-type code → label map |
| 4 | MySQL account query |
| 5 | MySQL subscription query |
| 6 | OneBill token manager |
| 7 | Account payload builder |
| 8 | Subscription payload builder |
| 9 | Combined per-account worker (account → subscriptions) |
| 10 | Parallel migration driver |
| 11 | Run + results, failures, error summaries |

## 1. Imports and Configuration

All tunables and external endpoints live in one place. The expected `.env` keys are the
superset of both source notebooks:

| Variable | Purpose |
|---|---|
| `CRM_TENANT_ID` / `CRM_CLIENT_ID` / `CRM_CLIENT_SECRET` | Azure AD app for Dataverse |
| `CRM_ENVIRONMENT_URL` | Dataverse env URL, no trailing slash |
| `DB_USERNAME` / `DB_PASSWORD` / `DB_HOST` | MySQL access |
| `CLIENT_ID` / `CLIENT_SECRET` / `API_USERNAME` / `API_PASSWORD` | OneBill OAuth |
| `CREATION_PROXY_ACCOUNT_NUMBER` | OneBill proxy account header value / account-number suffix |

`load_dotenv(override=True)` makes the `.env` authoritative over any pre-existing shell env.

`TEST_ROW_LIMIT` mirrors the `.head(5)` testing limiter used in both source notebooks —
bump it, or set it to `None`, once you're ready for a full run.

In [134]:
# %pip install msal mysql-connector-python sqlalchemy python-dotenv requests pandas

import os
import json
import time
import logging
import threading
import urllib.parse
from datetime import datetime, timedelta
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
import pandas as pd
from sqlalchemy import create_engine
from msal import ConfidentialClientApplication
from dotenv import load_dotenv

load_dotenv(override=True)

# --- Dataverse (Dynamics CRM) ---
CRM_TENANT_ID       = os.environ["CRM_TENANT_ID"]
CRM_CLIENT_ID       = os.environ["CRM_CLIENT_ID"]
CRM_CLIENT_SECRET   = os.environ["CRM_CLIENT_SECRET"]
CRM_ENVIRONMENT_URL = os.environ["CRM_ENVIRONMENT_URL"]

# --- MySQL ---
BI_DATASTORE_URL = (
    f"mysql+mysqlconnector://{os.environ['DB_USERNAME']}:{os.environ['DB_PASSWORD']}"
    f"@{os.environ['DB_HOST']}/bi_datastore"
)

# --- OneBill ---
ONEBILL_BASE_URL       = "https://sandbox-sg.onebillsoftware.com"
ONEBILL_TOKEN_URL      = f"{ONEBILL_BASE_URL}/oauth/token"
ONEBILL_ACCOUNT_URL    = f"{ONEBILL_BASE_URL}/rest/SubscriberService/v1/subscriber"
ONEBILL_SUBSCRIBER_URL = f"{ONEBILL_BASE_URL}/rest/SubscriberService/v1/subscribers"  # + /{accountNumber}
ONEBILL_ORDER_URL      = f"{ONEBILL_BASE_URL}/rest/OrderService/v1/order"
ONEBILL_PROXY_ACCT     = os.environ["CREATION_PROXY_ACCOUNT_NUMBER"]

# --- Migration tunables ---
MAX_WORKERS        = 20
TOKEN_TTL_FALLBACK = 3500   # seconds; used only if OAuth response omits expires_in

# Use this to limit rows while testing. Set to None once ready for a full run.
TEST_ROW_LIMIT = 5
TEST_RUN_NUMBER = "6"

# --- Defaults for blank contact fields ---
DEFAULT_FIRST_NAME = "John"
DEFAULT_LAST_NAME  = "Doe"
DEFAULT_EMAIL      = "someone@example.com"

# --- Fixed order fields (per current OneBill catalog — only one plan exists today) ---
DEFAULT_ORDER_STATE    = "1005"
DEFAULT_PRODUCT_NAME   = "Wholesale Fibre BS2 (Chorus)"
DEFAULT_PRICEPLAN_NAME = "WS Tail+Data - BS2 Res (Chorus) - 100/20"
DEFAULT_ACTION_TYPE    = "New"
DEFAULT_QUANTITY       = 1

# --- Logging ---
log_filename = f'full_migration_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log'
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.FileHandler(log_filename), logging.StreamHandler()],
)
logger = logging.getLogger(__name__)

2026-07-14 05:27:49,797 [WARNING] python-dotenv could not parse statement starting at line 1
2026-07-14 05:27:49,799 [WARNING] python-dotenv could not parse statement starting at line 5
2026-07-14 05:27:49,802 [WARNING] python-dotenv could not parse statement starting at line 12
2026-07-14 05:27:49,805 [WARNING] python-dotenv could not parse statement starting at line 16
2026-07-14 05:27:49,808 [WARNING] python-dotenv could not parse statement starting at line 22


## 2. Fetch Contacts from Dataverse

Pulls every active contact whose parent account has `accountcategorycode = 1` and a
non-null `accountnumber`. Uses **keyset pagination** on `contactid` (Dataverse's
`paging-cookie` mechanism can silently fail when combined with link-entity joins).

Unchanged from `OneBill_Customer_Migration.ipynb`.

In [135]:
# {extra_condition} is filled in per-page with a `contactid gt <last_seen>` clause
FETCHXML_TEMPLATE = """
<fetch version="1.0" output-format="xml-platform" mapping="logical" no-lock="false" count="5000">
  <entity name="contact">
    <attribute name="fullname"/>
    <attribute name="emailaddress1"/>
    <attribute name="telephone1"/>
    <attribute name="contactid"/>
    <attribute name="vgr_contacttypes"/>
    <attribute name="mobilephone"/>
    <attribute name="firstname"/>
    <attribute name="lastname"/>
    <attribute name="vgr_contactcode"/>
    <order attribute="contactid" descending="false"/>
    <filter type="and">
        <condition attribute="parentcustomerid" operator="ne" value="7378af87-be17-eb11-a813-000d3a7940d5" uiname="Portal Default Account" uitype="account"/>
        {extra_condition}
    </filter>
    <link-entity name="account" from="accountid" to="parentcustomerid" link-type="inner" alias="AccountCode">
      <attribute name="accountnumber"/>
      <filter type="and">
        <condition attribute="vgr_datasource" operator="eq" value="vBill"/>
      </filter>
    </link-entity>
  </entity>
</fetch>
""".strip()

# The column the link-entity emits — we rename it to AccountCode after the fetch
LINKED_ACCOUNTNUMBER_COL = "AccountCode.accountnumber"


def get_dataverse_token() -> str:
    # OAuth2 client-credentials bearer token for the Dataverse environment.
    app = ConfidentialClientApplication(
        client_id=CRM_CLIENT_ID,
        client_credential=CRM_CLIENT_SECRET,
        authority=f"https://login.microsoftonline.com/{CRM_TENANT_ID}",
    )
    result = app.acquire_token_for_client(scopes=[f"{CRM_ENVIRONMENT_URL}/.default"])
    if "access_token" not in result:
        raise RuntimeError(f"Token acquisition failed: {result.get('error_description')}")
    return result["access_token"]


def get_contacts(token: str, max_pages: int = 50) -> pd.DataFrame:
    # Fetch all contacts from Dataverse using keyset pagination on contactid.
    headers = {
        "Authorization":    f"Bearer {token}",
        "OData-MaxVersion": "4.0",
        "OData-Version":    "4.0",
        "Accept":           "application/json",
        "Prefer":           "odata.maxpagesize=5000",
    }

    all_records: list[dict] = []
    last_contactid: str | None = None
    page = 1

    while True:
        if last_contactid is None:
            extra_condition = ""
        else:
            extra_condition = (
                f'<condition attribute="contactid" operator="gt" value="{last_contactid}"/>'
            )

        fetch = FETCHXML_TEMPLATE.format(extra_condition=extra_condition)
        url = f"{CRM_ENVIRONMENT_URL}/api/data/v9.2/contacts?fetchXml={urllib.parse.quote(fetch)}"

        response = requests.get(url, headers=headers, timeout=60)
        response.raise_for_status()
        records = response.json().get("value", [])

        if not records:
            break

        all_records.extend(records)
        new_last = records[-1]["contactid"]
        logger.info(
            f"Contacts page {page}: fetched {len(records):,} "
            f"(total so far: {len(all_records):,})"
        )

        if len(records) < 5000:
            break

        if new_last == last_contactid:
            logger.warning("contactid did not advance — stopping to avoid infinite loop")
            break

        last_contactid = new_last
        page += 1

        if page > max_pages:
            logger.warning(f"Hit max_pages safety limit ({max_pages})")
            break

    df = pd.DataFrame(all_records)
    logger.info(f"Done — {len(df):,} contacts loaded")
    return df

In [136]:
dataverse_token = get_dataverse_token()
df_contacts = get_contacts(dataverse_token)

# Drop OData noise, keep only the columns we use
df_contacts = df_contacts.drop(
    columns=[c for c in ["@odata.etag", "fullname"] if c in df_contacts.columns],
    errors="ignore",
)

df_contacts = df_contacts.rename(columns={
    "vgr_contactcode":         "ContactCode",
    "vgr_contacttypes":        "Dynamics_ContactTypes",
    "telephone1":              "PhoneWork",
    "mobilephone":             "PhoneMobile",
    "emailaddress1":           "EmailAddresses",
    "firstname":               "FirstName",
    "lastname":                "LastName",
    LINKED_ACCOUNTNUMBER_COL:  "AccountCode",
})

# Substitute defaults for blank fields (rule: no blank first/last/email allowed)
df_contacts["FirstName"]      = df_contacts["FirstName"].replace("", pd.NA).fillna(DEFAULT_FIRST_NAME)
df_contacts["LastName"]       = df_contacts["LastName"].replace("", pd.NA).fillna(DEFAULT_LAST_NAME)
df_contacts["EmailAddresses"] = df_contacts["EmailAddresses"].replace("", pd.NA).fillna(DEFAULT_EMAIL)

df_contacts['BillingContact'] = df_contacts['ContactCode'].str.startswith('BILLING')

# df_contacts['AccountCode'] = df_contacts['AccountCode'] + '.' + TEST_RUN_NUMBER 

df_contacts['OneBill_ContactType'] = df_contacts['ContactCode'].apply(
    lambda x: '0' if str(x).startswith('BILLING') else '1'
)

logger.info(f"Contacts after default substitution: {len(df_contacts):,}")
df_contacts.head()

2026-07-14 05:27:57,707 [INFO] Contacts page 1: fetched 5,000 (total so far: 5,000)
2026-07-14 05:28:04,783 [INFO] Contacts page 2: fetched 5,000 (total so far: 10,000)
2026-07-14 05:28:11,798 [INFO] Contacts page 3: fetched 5,000 (total so far: 15,000)
2026-07-14 05:28:18,639 [INFO] Contacts page 4: fetched 5,000 (total so far: 20,000)
2026-07-14 05:28:39,827 [INFO] Contacts page 5: fetched 5,000 (total so far: 25,000)
2026-07-14 05:28:44,391 [INFO] Contacts page 6: fetched 5,000 (total so far: 30,000)
2026-07-14 05:28:49,426 [INFO] Contacts page 7: fetched 5,000 (total so far: 35,000)
2026-07-14 05:28:57,858 [INFO] Contacts page 8: fetched 5,000 (total so far: 40,000)
2026-07-14 05:29:02,215 [INFO] Contacts page 9: fetched 1,886 (total so far: 41,886)
2026-07-14 05:29:02,282 [INFO] Done — 41,886 contacts loaded
2026-07-14 05:29:02,397 [INFO] Contacts after default substitution: 41,886


,PhoneMobile,contactid,LastName,ContactCode,FirstName,EmailAddresses,AccountCode,PhoneWork,Dynamics_ContactTypes,BillingContact,OneBill_ContactType
0,021569156,7266d549-cbfa-ee11-9f89-000d3a6a0933,Prentice,C-00090123,Lynn,lynn.prentice@gmail.com,47621153,NaN,NaN,False,1
1,0212141547,3b4848b4-7c5b-f011-bec1-000d3a6a2a4e,Sibbe,C-00100775,Judy,sibbe@actrix.co.nz,99993083,NaN,NaN,False,1
2,N/A,83b8ffe3-c69b-ef11-8a69-000d3a6a332a,Rose,C-00094944,Angel,accounts@techspanonline.com,99999381,6498276567,NaN,False,1
3,0210464741,0abb6f81-7d3d-ef11-a316-000d3a6a3623,Matthews,C-00092237,Angela,accounts@agedadvisor.co.nz,99937383,NaN,NaN,False,1
4,NaN,7b967bab-6c69-ef11-a670-000d3a6a3b2b,Clarke,C-00093561,Andre,andre.clarke@hbtech.co.nz,99993096,NaN,287790008,False,1


## 3. Contact-type code → label map, and contact indexing

`vgr_contacttypes` is a comma-separated list of Dataverse option codes. Each code maps
to its OneBill label and becomes an `associateValues` entry under the `Dynamics Contact
Types` attribute, with `sequence` preserving order.

Contacts are then grouped by `AccountCode`. Within each account, the `BILLING-` contact
is placed first (OneBill derives the account's display name from the first contact in
the payload).

In [137]:
CONTACT_TYPE_MAP = {
    "287790000": "Billing",
    "287790001": "Technical",
    "287790002": "Outage - Email",
    "287790009": "Outage - SMS",
    "287790003": "Primary",
    "287790004": "Technical - Data",
    "287790005": "Technical - Voice",
    "287790006": "Commercial",
    "287790008": "Communication",
    "287790007": "Voyager Staff",
}


def parse_contact_types(raw) -> list[str]:
    """Parse a vgr_contacttypes value into an ordered list of OneBill labels."""
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return []
    codes = [c.strip() for c in str(raw).split(",") if c.strip()]
    return [CONTACT_TYPE_MAP[c] for c in codes if c in CONTACT_TYPE_MAP]

In [138]:
def index_contacts_by_account(df: pd.DataFrame) -> dict[str, list[dict]]:
    """Group contact rows by AccountCode. Returns {AccountCode: [contact_dict, ...]}.

    Within each account, the BILLING- contact is placed FIRST. OneBill derives
    the account-level display name from the first contact in the payload, so
    putting the billing contact (ContactCode starts with 'BILLING') at index 0
    ensures the UI shows the right name. Stable sort preserves the original
    order among non-billing contacts (and among billing contacts if, for
    legacy-data reasons, there's more than one).
    """
    by_account: dict[str, list[dict]] = defaultdict(list)
    for _, row in df.iterrows():
        acct = row.get("AccountCode")
        if pd.isna(acct) or acct in (None, ""):
            continue
        by_account[str(acct)].append(row.to_dict())

    # Sort each account's contacts: BillingContact=True first, then everyone else.
    for acct, contacts in by_account.items():
        contacts.sort(key=lambda c: not bool(c.get("BillingContact", False)))

    return dict(by_account)


contacts_by_account = index_contacts_by_account(df_contacts)
logger.info(
    f"Indexed {sum(len(v) for v in contacts_by_account.values()):,} contacts "
    f"across {len(contacts_by_account):,} accounts"
)

2026-07-14 05:29:04,231 [INFO] Indexed 41,878 contacts across 37,552 accounts


## 4. MySQL Account Query

Pulls every `vBill` account from `bi_datastore.billing_account`, cleans stray suffixes
out of `AccountName` (e.g. `(BOND)`, `(LIQUIDATION)`), and classifies each account as
Individual (`1001`) or Business (`1002`) for OneBill's `accountType`.

`AccountCode_Batch` is the account number that will actually be sent to OneBill —
`{AccountCode}_{CREATION_PROXY_ACCOUNT_NUMBER}` — and it's what subscriptions will be
matched against later.

Unchanged from `OneBill_Customer_Migration.ipynb`.

In [ ]:
ACCOUNT_QUERY = """
SELECT
    `AccountName` AS `AccountName_Original`
    ,TRIM(
		REPLACE(
			REPLACE(
				REPLACE(
					REPLACE(
						REPLACE(
							REPLACE(
								REPLACE(
									REPLACE(
										REPLACE(
											REPLACE(
												REPLACE(
													REPLACE(
														REPLACE(
															REPLACE(
																REPLACE(
																	REPLACE(
																		REPLACE(
																			REPLACE(
																				REPLACE(
																					REPLACE(`AccountName`, '(BOND - DECLINED)', ''),
																				'(BOND)', ''),
																			'(ICMS)', ''),
																		'(Staff)', ''),
																	'(X)', ''),
																'(In Liquidation)', ''),
															'zz-', ''),
														'(DECLINED)', ''),
													'(Operator)', ''),
												'(Liquidation )', ''),
											'(LIQUIDATION)', ''),
										'(Under Liquidation)', ''),
									'[LIQUIDATION]', ''),
								'(Bad - Debt)', ''),
							'(Bad Debt)', ''),
						'(Bad-Debt)', ''),
					'(BOND - DECLINED)', ''),
				'(BOND DECLINED)', ''),
			'(BOND-DECLINED)', ''),
        '(COMPRIMISED)', '') -- Even though it is spelt incorrectly there are no '(COMPROMISED)' accounts, only '(COMPRIMISED)', so we need to catch this too
    ) AS `AccountName_Cleaned`
    ,`AccountCode`
    ,`CreatedDate`
    ,`ClosedDate`
    ,`AccountType`
    ,CASE
		WHEN `AccountType` IN ('Residential', 'Actrix Residential', 'Consumer', 'Standard Account', 'Internal-Use Account', 'Staff')
        AND `AccountName` NOT LIKE '%Ltd%'
        AND `AccountName` NOT LIKE '%Limited%'
        AND `AccountName` NOT LIKE '%Pty%'
		THEN '1001' -- Individual Customer
        ELSE '1002' -- Business Customer 
	END AS `OneBill_AccountType`
    ,CASE
        WHEN `_temporary_crmonly_addr1` IS NULL OR `_temporary_crmonly_addr1` = '' THEN '1 Somewhere Place'
        ELSE `_temporary_crmonly_addr1`
    END AS `Address1`
    ,`_temporary_crmonly_addr2` AS `Address2`
    ,`_temporary_crmonly_suburb` AS `Suburb`
    ,CASE
        WHEN `_temporary_crmonly_city` IS NULL OR `_temporary_crmonly_city` = '' THEN 'Auckland'
        ELSE `_temporary_crmonly_city`
    END AS `City`
    ,CASE
        WHEN `_temporary_crmonly_postcode` IS NULL OR `_temporary_crmonly_postcode` = '' THEN '0001'
        ELSE `_temporary_crmonly_postcode`
    END AS `Postcode`
    ,`_temporary_crmonly_dob` AS `DateOfBirth`
FROM
    bi_datastore.billing_account
WHERE
    `_DataSource` = 'vBill'
AND 
	`AccountCode` = '99965692'
ORDER BY
    `AccountCode` DESC
""".strip()

engine = create_engine(BI_DATASTORE_URL)
df_accounts = pd.read_sql(ACCOUNT_QUERY, con=engine)

# AccountCode as string for consistent dict lookups against Dataverse-side strings
df_accounts["AccountCode"] = df_accounts["AccountCode"].astype(str)

df_accounts['AccountName_Unique'] = df_accounts['AccountName_Cleaned'] + ' (' + df_accounts['AccountCode'] + ')'
df_accounts['AccountCode_Batch'] = df_accounts['AccountCode'] + '_' + os.environ["CREATION_PROXY_ACCOUNT_NUMBER"] + '.' + TEST_RUN_NUMBER

logger.info(f"Loaded {len(df_accounts):,} accounts from MySQL")

#df_accounts = df_accounts.head(5) #Use this to limit the number of accounts we attempt to migrate while testing. Remove or increase the number when ready for a larger migration.

df_accounts.head()

2026-07-14 05:29:09,681 [INFO] Loaded 1 accounts from MySQL


,AccountName_Original,AccountName_Cleaned,AccountCode,CreatedDate,ClosedDate,AccountType,OneBill_AccountType,Address1,Address2,Suburb,City,Postcode,DateOfBirth,AccountName_Unique,AccountCode_Batch
0,Williams Internet Limited,Williams Internet Limited,99965692,2025-11-19,None,Wholesale Unlimited,1002,124 Peterborough Street\nChristchurch Central ...,None,None,Auckland,0001,None,Williams Internet Limited (99965692),99965692_import01.6


## 5. MySQL Subscription Query

Pulls every subscription from `bi_curated_views.reporting_subscription` and computes
`Proxy_AccountCode` the same way accounts do, so the two datasets join cleanly on the
OneBill account number. Subscriptions are then grouped by `AccountCode` so an account
with multiple subscriptions gets all of them created.

Unchanged from `OneBill_Subscription_Migration.ipynb`, aside from indexing by account.

In [140]:
SUBSCRIPTION_QUERY = """
SELECT 
	subs.*
	,acct.BillingDay
FROM bi_curated_views.reporting_subscription subs
LEFT JOIN bi_curated_views.reporting_account acct
ON subs.AccountCode = acct.AccountCode
WHERE subs.`AccountCode` = '99965692'
ORDER BY RAND();
""".strip()

df_subscriptions = pd.read_sql(SUBSCRIPTION_QUERY, con=engine)
logger.info(f"Loaded {len(df_subscriptions):,} subscriptions from MySQL")

df_subscriptions["AccountCode"] = df_subscriptions["AccountCode"].astype(str)
df_subscriptions['Proxy_AccountCode'] = df_subscriptions['AccountCode'] + '_' + ONEBILL_PROXY_ACCT + '.' + TEST_RUN_NUMBER

if TEST_ROW_LIMIT is not None:
    df_subscriptions = df_subscriptions.head(TEST_ROW_LIMIT)  # Testing limiter — remove/raise for a full run.

df_subscriptions = df_subscriptions.head()
df_subscriptions

2026-07-14 05:29:19,120 [INFO] Loaded 323 subscriptions from MySQL


,_rowmodified,_pk,AccountCode,ServiceType,SubscriptionUSN,SubscriptionLabel,SubscriptionStartDate,SubscriptionEndDate,PlanCode,PlanStartDate,...,BillingDay,CostCentre,Supplier,SupplierServiceID,SupplierAccessID,CircuitType,Server,CustomerSuppliedReference,BillingDay,Proxy_AccountCode
0,2026-07-03 20:22:12,4308726386,99965692,Broadband - Fibre,V113085690,V113085690_0@williamsinternet.com,2026-04-24,2026-06-17,SC-04045,2026-04-24,...,31,None,Enable,ENVOYB02643373,None,UFB 100/20/2.5/2.5,None,Managed by Williams Limited,31,99965692_import01.6
1,2026-07-03 20:37:21,4241229293,99965692,Broadband - Fibre,V113074108,7.128salisbury@williamsinternet.com,2026-02-16,None,SC-04045,2026-02-16,...,31,None,Enable,ENVOYB02630859,None,UFB 100/20/2.5/2.5,None,Managed by Williams LTD,31,99965692_import01.6
2,2026-07-03 20:37:09,4213179661,99965692,Broadband - Fibre,V113069066,22.200carrington@williamsinternet.com,2026-01-19,None,SC-02140,2026-01-19,...,31,None,Chorus,1643216783,None,UFB 100/20/2.5/2.5,None,Williams Corp - AKL,31,99965692_import01.6
3,2026-07-03 20:36:57,4175483384,99965692,Broadband - Fibre,V113064471,1643180631@no-username,2025-12-09,None,SC-04046,2025-12-09,...,31,None,Chorus,1643180631,None,UFB 100/20/2.5/2.5,None,WC CHCH T9,31,99965692_import01.6
4,2026-07-03 20:22:20,4277017370,99965692,Broadband - Fibre,V113080543,4.245kilmore@williamsinternet.com,2026-03-23,None,SC-04045,2026-03-23,...,31,None,Enable,ENVOYB02636767,None,UFB 100/20/2.5/2.5,None,Managed by Williams Limited,31,99965692_import01.6


In [141]:
def index_subscriptions_by_account(df: pd.DataFrame) -> dict[str, list[dict]]:
    """Group subscription rows by AccountCode. Returns {AccountCode: [subscription_dict, ...]}."""
    by_account: dict[str, list[dict]] = defaultdict(list)
    for _, row in df.iterrows():
        acct = row.get("AccountCode")
        if pd.isna(acct) or acct in (None, ""):
            continue
        by_account[str(acct)].append(row.to_dict())
    return dict(by_account)


subscriptions_by_account = index_subscriptions_by_account(df_subscriptions)
logger.info(
    f"Indexed {sum(len(v) for v in subscriptions_by_account.values()):,} subscriptions "
    f"across {len(subscriptions_by_account):,} accounts"
)

2026-07-14 05:29:19,149 [INFO] Indexed 5 subscriptions across 1 accounts


## 6. OneBill OAuth Token Manager

Thread-safe bearer token cache with proactive refresh (100s early to absorb clock skew
and in-flight requests). Shared by both account-creation and order-creation calls.

In [142]:
class TokenManager:
    """Thread-safe bearer token cache with proactive refresh."""

    def __init__(self):
        self._lock = threading.Lock()
        self._token: str | None = None
        self._expires_at: datetime = datetime.min

    def get_token(self) -> str:
        with self._lock:
            if datetime.now() >= self._expires_at:
                self._refresh()
            return self._token

    def _refresh(self) -> None:
        logger.info("Refreshing OneBill OAuth token...")
        token_data = {
            "grant_type":    "password",
            "client_id":     os.environ["CLIENT_ID"],
            "client_secret": os.environ["CLIENT_SECRET"],
            "username":      os.environ["API_USERNAME"],
            "password":      os.environ["API_PASSWORD"],
        }
        response = requests.post(
            ONEBILL_TOKEN_URL,
            data=token_data,
            headers={"Content-Type": "application/x-www-form-urlencoded"},
            timeout=30,
        )
        response.raise_for_status()
        payload = response.json()
        self._token = payload["access_token"]
        ttl = payload.get("expires_in", TOKEN_TTL_FALLBACK)
        # Refresh 100 s early to absorb clock skew + in-flight requests
        self._expires_at = datetime.now() + timedelta(seconds=ttl - 100)
        logger.info("Token valid until %s", self._expires_at.strftime("%H:%M:%S"))


token_manager = TokenManager()

## 7. Account Payload Builder + Create Call

Builds the OneBill account-creation payload (address, contacts, account attributes) and
POSTs it to `SubscriberService/v1/subscriber`.

**Idempotency:** `create_onebill_account` returns one of three statuses instead of just
raising on any non-successful validation:

- `"created"` — brand-new account.
- `"exists"` — OneBill's validation message contains `"already exists"`; **not** treated
  as a failure, so the pipeline proceeds straight to subscription creation.
- `"failed"` — any other validation error; subscription creation is skipped for this
  account.

In [143]:
def _clean(value):
    """Return None for blanks/NaN; pass everything else through unchanged."""
    if value is None:
        return None
    if isinstance(value, float) and pd.isna(value):
        return None
    if isinstance(value, str) and value.strip() == "":
        return None
    return value


def build_communication_points(contact: dict) -> list[dict]:
    """Emit Email + Phone communication points, skipping blanks. Mobile preferred over work phone."""
    points: list[dict] = []

    email = _clean(contact.get("EmailAddresses"))
    if email is not None:
        points.append({"type": "EMAIL", "value": str(email)})

    # Prefer mobile, fall back to work phone (NaN-safe)
    phone = _clean(contact.get("PhoneMobile")) or _clean(contact.get("PhoneWork"))
    if phone is not None:
        points.append({"type": "Phone", "value": str(phone)})

    return points


def build_contact_block(contact: dict) -> dict:
    """Build a single OneBill contact entry from a Dynamics contact row."""
    block = {
        "firstName":          contact["FirstName"],
        "lastName":           contact["LastName"],
        'contactType':        contact.get("OneBill_ContactType"),
        'primaryContact':     contact.get("BillingContact", False),
        'billingContact':     contact.get("BillingContact", False),
        "communicationPoint": build_communication_points(contact),
    }

    types = parse_contact_types(contact.get("Dynamics_ContactTypes"))
    if types:
        block["contactAttributes"] = [
            {
                "key":                   "Dynamics Contact Types",
                "value":                 types[0],
                "multipleEntriesConfig": "ENABLED",
                "attributeValuesInfo": {
                    "associateValues": [
                        {"value": t, "sequence": i + 1}
                        for i, t in enumerate(types)
                    ],
                },
            },
            {
                "key": "Contact Code",
                "value": contact["ContactCode"]
            }
        ]

    return block


def build_placeholder_contact() -> dict:
    """A single safe contact used when an account has zero Dynamics contacts."""
    return {
        "firstName": DEFAULT_FIRST_NAME,
        "lastName":  DEFAULT_LAST_NAME,
        "communicationPoint": [
            {"type": "EMAIL", "value": DEFAULT_EMAIL},
        ],
    }


def build_account_payload(row: pd.Series, contacts_by_account: dict[str, list[dict]]) -> str:
    """Build the OneBill account-creation payload for a single MySQL account row."""
    # Convert pandas row -> dict, then NaN -> None everywhere (json.dumps emits
    # 'NaN' for float NaN which OneBill won't accept as valid JSON)
    row = {k: _clean(v) if not isinstance(v, (list, dict)) else v for k, v in row.to_dict().items()}

    # --- Resolve contacts for this account
    extra_contacts = contacts_by_account.get(str(row["AccountCode"]), [])
    contact_list = [build_contact_block(c) for c in extra_contacts]
    if not contact_list:
        contact_list = [build_placeholder_contact()]

    def serialize_date(value, fmt=None):
        if value is None:
            return None
        if hasattr(value, 'isoformat'):
            return value.strftime(fmt) if fmt else value.isoformat()
        if fmt:
            try:
                return datetime.strptime(str(value), '%Y-%m-%d').strftime(fmt)
            except ValueError:
                return str(value)
        return str(value)

    # --- Address (single element list as per the example payload)
    address_block = {
        "addLine1":        row["Address1"],
        "addLine2":        row.get("Address2"),
        "city":            row["City"],
        "state":           None,
        "country":         "New Zealand",
        "zip":             str(row["Postcode"]) if row.get("Postcode") is not None else None,
        "defaultShipping": True,
        "defaultBilling":  True,
    }

    payload = {
        "accountType":   row['OneBill_AccountType'],
        "accountName":   row["AccountName_Cleaned"],
        'accountNumber': row["AccountCode_Batch"],
        'accountingDisplayName': "",
        'activationStartDate': row["CreatedDate"].strftime("%Y-%m-%d"),
        "address":       [address_block],
        "contact":       contact_list,
        "accountAttribute": [
        {
            "key":   "vBill Account Types",
            "value": row.get("AccountType"),
        },
        {
            'key': 'Date Of Birth',
            'value': serialize_date(row.get("DateOfBirth"), "%d/%m/%Y")
        },
        {
            'key': 'vBill Account Name',
            'value': row.get("AccountName_Original")
        }]
    }

    return json.dumps(payload)

In [144]:
# Substring OneBill uses in validationErrorInfo when the account already exists.
# Matched case-insensitively so this doesn't need exact-wording maintenance.
ACCOUNT_ALREADY_EXISTS_MARKER = "already exists"


def create_onebill_account(session: requests.Session, payload: str) -> tuple[str, dict | None, str | None]:
    """POST one account to OneBill.

    Returns (status, response_json_or_None, error_message_or_None) where
    status is one of "created", "exists", "failed".

    "exists" is NOT an error — it means the account was already present in
    OneBill (e.g. from a previous run of this notebook), so migration should
    proceed straight to subscription creation for it.
    """
    headers = {"Authorization": f"Bearer {token_manager.get_token()}"}

    response = session.post(ONEBILL_ACCOUNT_URL, headers=headers, data=payload, timeout=30)
    response.raise_for_status()
    data = response.json()

    validation = data.get("validationResponse", {})
    if not validation.get("successful", True):
        errors   = validation.get("validationErrorInfo", [])
        messages = "; ".join(e.get("message", "") for e in errors)
        messages = messages or "validationResponse.successful = false"

        if ACCOUNT_ALREADY_EXISTS_MARKER in messages.lower():
            return "exists", data, messages

        return "failed", data, messages

    return "created", data, None

## 8. Subscription Payload Builder + Create Call

Looks up the account's `defaultShipping` address id in OneBill (cached per account
number), builds the order payload, and POSTs it to `OrderService/v1/order`.

The address lookup works the same way whether the account was just created in this run
or already existed beforehand — it simply reads back whatever OneBill currently has for
that account number.

Unchanged from `OneBill_Subscription_Migration.ipynb`.

In [145]:
_ship_address_cache: dict[str, str] = {}
_ship_address_cache_lock = threading.Lock()


def get_default_ship_address_id(session: requests.Session, account_number: str) -> str:
    """Return the account's defaultShipping address id, fetching + caching on first use."""
    with _ship_address_cache_lock:
        cached = _ship_address_cache.get(account_number)
    if cached is not None:
        return cached

    headers = {"Authorization": f"Bearer {token_manager.get_token()}"}
    url = f"{ONEBILL_SUBSCRIBER_URL}/{account_number}"
    response = session.get(url, headers=headers, timeout=30)
    response.raise_for_status()
    detail = response.json()

    addresses = detail.get("address", [])
    default_address = next((a for a in addresses if a.get("defaultShipping")), None)
    if default_address is None:
        raise ValueError(f"No defaultShipping address found for account {account_number}")

    ship_add_id = str(default_address["id"])
    with _ship_address_cache_lock:
        _ship_address_cache[account_number] = ship_add_id
    return ship_add_id

In [146]:
def _to_iso_midnight(value) -> str | None:
    """Format a date/datetime/str value as OneBill's 'YYYY-MM-DDT00:00:00'. None-safe."""
    if value is None or pd.isna(value):
        return None
    ts = pd.Timestamp(value)
    return ts.strftime("%Y-%m-%dT00:00:00")


def _beginning_of_this_month() -> str:
    """Return today's month, day 1, at midnight, in OneBill's date format."""
    today = datetime.now()
    return today.replace(day=1).strftime("%Y-%m-%dT00:00:00")


def build_subscription_payload(subscription: dict, ship_add_id: str) -> dict:
    """Build the OneBill order-create payload for a single subscription row. No transforms."""
    return {
        "accountNumber":      str(subscription["Proxy_AccountCode"]),
        "orderState":         DEFAULT_ORDER_STATE,
        "billThissOrder":     False,
        "isSkipProvisioning": True,
        "orderElement": [
            {
                "quantity":               subscription["Quantity"],
                "actionType":             DEFAULT_ACTION_TYPE,
                "fulfilledDate":          _to_iso_midnight(subscription["SubscriptionStartDate"]),
                "recurringStartDate":     _beginning_of_this_month(),
                "subscriptionIdentifier": str(subscription["SubscriptionUSN"]),
                "productName":            DEFAULT_PRODUCT_NAME,
                "priceplanName":          DEFAULT_PRICEPLAN_NAME,
                "shipAddId":              ship_add_id,
            }
        ],
        "orderElementAttribute": [
                {
                    "featureName": "Radius Username",
                    "type": "0",
                    "value": str(subscription["SubscriptionLabel"]),
                },
                {
                    "featureName": "External Service ID",
                    "value": subscription["SupplierServiceID"]
                }
        ]
    }


def create_onebill_order(session: requests.Session, payload: dict) -> dict:
    """POST one subscription/order to OneBill. Raises ValueError on validation failure."""
    headers = {"Authorization": f"Bearer {token_manager.get_token()}"}

    response = session.post(ONEBILL_ORDER_URL, headers=headers, json=payload, timeout=30)
    response.raise_for_status()
    data = response.json()

    validation = data.get("validationResponse", {})
    if not validation.get("successful", True):
        errors   = validation.get("validationErrorInfo", [])
        messages = "; ".join(e.get("message", "") for e in errors)
        raise ValueError(messages or "validationResponse.successful = false")

    return data

## 9. Combined Per-Account Worker

`migrate_account` does both steps for a single account row: create/confirm the account,
then (if it exists in OneBill one way or another) create every subscription that belongs
to it. `migrate_subscription` handles one subscription's address lookup + build + POST.

**Linking a subscription to its OneBill account:** `migrate_subscription` uses the
subscription's own `Proxy_AccountCode` (set when subscriptions were loaded in Section 5)
as the single source of truth for which OneBill account it belongs to. That same value is
used both to look the account up in OneBill (`get_default_ship_address_id`, for the
shipping address needed on the order) and as the `accountNumber` in the order payload
itself — so there's no risk of the lookup and the payload pointing at two different
account numbers.

In [147]:
def migrate_subscription(
    session: requests.Session,
    subscription: dict,
) -> dict:
    """Resolve shipAddId, build, and POST a single subscription. Returns a result dict.

    The subscription's own `Proxy_AccountCode` (built when subscriptions were loaded)
    is the single source of truth for which OneBill account this subscription belongs
    to — it's used both to look the account up in OneBill (for the shipAddId) and as
    the `accountNumber` in the order payload itself.
    """
    account_number  = str(subscription["Proxy_AccountCode"])
    subscription_id = subscription["SubscriptionUSN"]

    t_lookup, t_build, t_net = 0.0, 0.0, 0.0
    status, error, order_id, ship_add_id = "failed", None, None, None

    try:
        t0 = time.perf_counter()
        ship_add_id = get_default_ship_address_id(session, account_number)
        t_lookup = time.perf_counter() - t0

        t1 = time.perf_counter()
        payload = build_subscription_payload(subscription, ship_add_id)
        t_build = time.perf_counter() - t1

        t2 = time.perf_counter()
        response = create_onebill_order(session, payload)
        t_net = time.perf_counter() - t2

        order_id = response.get("orderId", "unknown")
        status = "success"
        logger.info(
            f"    [OK] subscription {subscription_id} (shipAddId={ship_add_id}, "
            f"OneBill orderId={order_id}) — lookup={t_lookup*1000:.0f}ms "
            f"build={t_build*1000:.0f}ms net={t_net*1000:.0f}ms"
        )

    except Exception as e:
        error = str(e)
        logger.error(f"    [FAIL] subscription {subscription_id} — {error}")

    return {
        "AccountCode":       account_number,
        "SubscriptionUSN":   subscription_id,
        "shipAddId":         ship_add_id,
        "status":            status,
        "onebill_order_id":  order_id,
        "error":             error,
        "elapsed_lookup_ms": round(t_lookup * 1000, 1),
        "elapsed_build_ms":  round(t_build * 1000, 1),
        "elapsed_net_ms":    round(t_net * 1000, 1),
    }

In [148]:
def migrate_account(
    row: pd.Series,
    session: requests.Session,
    contacts_by_account: dict[str, list[dict]],
    subscriptions_by_account: dict[str, list[dict]],
) -> dict:
    """Create (or confirm existence of) one account, then create its subscriptions.

    Returns a dict with the account-level result plus a nested list of
    subscription-level results.
    """
    account_code = row["AccountCode"]
    account_name = row["AccountName_Unique"]

    t0 = time.perf_counter()
    payload = build_account_payload(row, contacts_by_account)
    t_build = time.perf_counter() - t0

    t_net = 0.0
    account_status = "failed"
    account_error = None
    onebill_account_id = None

    try:
        t1 = time.perf_counter()
        acc_status, response, message = create_onebill_account(session, payload)
        t_net = time.perf_counter() - t1

        account_status = acc_status
        if acc_status == "created":
            onebill_account_id = (response or {}).get("accountId", "unknown")
            logger.info(
                f"  [OK] {account_code} (OneBill id={onebill_account_id}) — "
                f"build={t_build*1000:.0f}ms net={t_net*1000:.0f}ms"
            )
        elif acc_status == "exists":
            account_error = message
            logger.info(f"  [EXISTS] {account_code} already in OneBill — proceeding to subscriptions")
        else:  # failed
            account_error = message
            logger.error(f"  [FAIL] {account_code} — {account_error}")

    except Exception as e:
        if "t1" in locals():
            t_net = time.perf_counter() - t1
        account_error = str(e)
        logger.error(f"  [FAIL] {account_code} — {account_error}")

    account_result = {
        "AccountCode":      account_code,
        "AccountName":      account_name,
        "status":           account_status,   # created | exists | failed
        "onebill_id":       onebill_account_id,
        "error":            account_error,
        "elapsed_build_ms": round(t_build * 1000, 1),
        "elapsed_net_ms":   round(t_net * 1000, 1),
    }

    subscription_results: list[dict] = []

    # Only attempt subscriptions if the account exists in OneBill one way or
    # another (freshly created, or already there from a prior run).
    if account_status in ("created", "exists"):
        subs = subscriptions_by_account.get(str(account_code), [])
        if not subs:
            logger.info(f"  -- no subscriptions found for account {account_code}")
        for sub in subs:
            subscription_results.append(
                migrate_subscription(session, sub)
            )
    else:
        logger.info(f"  -- skipping subscriptions for {account_code} (account creation failed)")

    return {"account": account_result, "subscriptions": subscription_results}

## 10. Parallel Migration Driver

Runs `migrate_account` for every account row in a `ThreadPoolExecutor`, sharing one
`requests.Session`/connection pool and one `TokenManager` across workers — same pattern
as both source notebooks.

In [149]:
def migrate(
    df_accounts: pd.DataFrame,
    contacts_by_account: dict[str, list[dict]],
    subscriptions_by_account: dict[str, list[dict]],
    max_workers: int = MAX_WORKERS,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Migrate every account (and its subscriptions) in df_accounts, in parallel.

    Returns (accounts_results_df, subscriptions_results_df).
    """
    session = requests.Session()
    adapter = requests.adapters.HTTPAdapter(
        pool_connections=max_workers,
        pool_maxsize=max_workers,
    )
    session.mount("https://", adapter)
    session.headers.update({
        "proxy_accountNumber": ONEBILL_PROXY_ACCT,
        "Content-Type":        "application/json",
    })

    rows  = [row for _, row in df_accounts.iterrows()]
    total = len(rows)
    account_results: list[dict] = []
    subscription_results: list[dict] = []

    logger.info(f"Starting migration of {total:,} accounts with {max_workers} workers...")
    wall_start = time.perf_counter()

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(
                migrate_account, row, session, contacts_by_account, subscriptions_by_account
            ): row["AccountCode"]
            for row in rows
        }

        for i, future in enumerate(as_completed(futures), start=1):
            result = future.result()
            account_results.append(result["account"])
            subscription_results.extend(result["subscriptions"])

            if i % 50 == 0 or i == total:
                ok      = sum(1 for r in account_results if r["status"] == "created")
                exists  = sum(1 for r in account_results if r["status"] == "exists")
                fail    = sum(1 for r in account_results if r["status"] == "failed")
                logger.info(f"Progress: {i}/{total} accounts — {ok} created, {exists} already existed, {fail} failed")

    wall_elapsed = time.perf_counter() - wall_start

    accounts_results_df = pd.DataFrame(account_results)
    subscriptions_results_df = pd.DataFrame(subscription_results)

    created = (accounts_results_df["status"] == "created").sum()
    existed = (accounts_results_df["status"] == "exists").sum()
    failed  = (accounts_results_df["status"] == "failed").sum()

    sub_ok = (subscriptions_results_df["status"] == "success").sum() if not subscriptions_results_df.empty else 0
    sub_fail = (subscriptions_results_df["status"] == "failed").sum() if not subscriptions_results_df.empty else 0

    logger.info(
        f"Migration done in {wall_elapsed:.1f}s — accounts: {created} created, "
        f"{existed} already existed, {failed} failed; subscriptions: {sub_ok} succeeded, "
        f"{sub_fail} failed. (log: {log_filename})"
    )

    # --- Profiling summary
    print("\n=== Profiling Summary ===")
    print(f"Total wall time:              {wall_elapsed:.1f}s")
    if wall_elapsed > 0:
        print(f"Throughput:                   {total / wall_elapsed:.1f} accounts/s")
    print(f"Accounts created / existed / failed: {created} / {existed} / {failed}")
    print(f"Subscriptions succeeded / failed:    {sub_ok} / {sub_fail}")
    print(f"Unique ship-address lookups cached:  {len(_ship_address_cache):,}")
    print("==========================")

    return accounts_results_df, subscriptions_results_df

## 11. Run Migration

In [150]:
accounts_results_df, subscriptions_results_df = migrate(
    df_accounts, contacts_by_account, subscriptions_by_account
)

account_failures = accounts_results_df[accounts_results_df["status"] == "failed"]
print(f"\nFailed account creations ({len(account_failures):,}):")
account_failures.head(20)

2026-07-14 05:29:19,290 [INFO] Starting migration of 1 accounts with 20 workers...
2026-07-14 05:29:19,293 [INFO] Refreshing OneBill OAuth token...
2026-07-14 05:29:21,091 [INFO] Token valid until 05:56:48
2026-07-14 05:29:23,364 [INFO]   [OK] 99965692 (OneBill id=unknown) — build=0ms net=4071ms
2026-07-14 05:29:25,467 [INFO]     [OK] subscription V113085690 (shipAddId=140507, OneBill orderId=unknown) — lookup=476ms build=0ms net=1626ms
2026-07-14 05:29:27,068 [INFO]     [OK] subscription V113074108 (shipAddId=140507, OneBill orderId=unknown) — lookup=0ms build=0ms net=1599ms
2026-07-14 05:29:28,989 [INFO]     [OK] subscription V113069066 (shipAddId=140507, OneBill orderId=unknown) — lookup=0ms build=0ms net=1920ms
2026-07-14 05:29:30,844 [INFO]     [OK] subscription V113064471 (shipAddId=140507, OneBill orderId=unknown) — lookup=0ms build=0ms net=1855ms
2026-07-14 05:29:32,721 [INFO]     [OK] subscription V113080543 (shipAddId=140507, OneBill orderId=unknown) — lookup=0ms build=0ms ne


=== Profiling Summary ===
Total wall time:              13.4s
Throughput:                   0.1 accounts/s
Accounts created / existed / failed: 1 / 0 / 0
Subscriptions succeeded / failed:    5 / 0
Unique ship-address lookups cached:  1

Failed account creations (0):


,AccountCode,AccountName,status,onebill_id,error,elapsed_build_ms,elapsed_net_ms


### Account failures — save to CSV

In [151]:
acct_out_path = f'Failed_Account_Migrations_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
account_failures.to_csv(acct_out_path, index=False)
print(f"Wrote {len(account_failures):,} account failures to {acct_out_path}")

Wrote 0 account failures to Failed_Account_Migrations_20260714_052932.csv


### Account failures — error summary

In [152]:
if not account_failures.empty:
    acct_error_summary = (
        account_failures.groupby("error")
        .agg(count=("AccountCode", "size"), example_account=("AccountCode", "first"))
        .sort_values("count", ascending=False)
        .reset_index()
    )
    print(f"Distinct account error messages: {len(acct_error_summary):,}")
else:
    print("No account failures to summarise.")
    acct_error_summary = pd.DataFrame()
acct_error_summary

No account failures to summarise.


""


### Subscription results

In [153]:
if not subscriptions_results_df.empty:
    subscription_failures = subscriptions_results_df[subscriptions_results_df["status"] == "failed"]
    print(f"\nFailed subscription creations ({len(subscription_failures):,}):")
else:
    subscription_failures = pd.DataFrame()
    print("\nNo subscriptions were attempted.")
subscription_failures.head(20)


Failed subscription creations (0):


,AccountCode,SubscriptionUSN,shipAddId,status,onebill_order_id,error,elapsed_lookup_ms,elapsed_build_ms,elapsed_net_ms


### Subscription failures — save to CSV

In [154]:
if not subscription_failures.empty:
    sub_out_path = f'Failed_Subscription_Migrations_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
    subscription_failures.to_csv(sub_out_path, index=False)
    print(f"Wrote {len(subscription_failures):,} subscription failures to {sub_out_path}")
else:
    print("No subscription failures to write.")

No subscription failures to write.


### Subscription failures — error summary

In [155]:
if not subscription_failures.empty:
    sub_error_summary = (
        subscription_failures.groupby("error")
        .agg(count=("SubscriptionUSN", "size"),
             example_account=("AccountCode", "first"),
             example_subscription=("SubscriptionUSN", "first"))
        .sort_values("count", ascending=False)
        .reset_index()
    )
    print(f"Distinct subscription error messages: {len(sub_error_summary):,}")
else:
    print("No subscription failures to summarise.")
    sub_error_summary = pd.DataFrame()
sub_error_summary

No subscription failures to summarise.


""
